In [2]:
import os
import dotenv
from langchain.chains.conversation.base import ConversationChain
from langchain.chains.llm import LLMChain
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=40
)

一、接口的最底层 ChatMessageHistory

In [3]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()
history.add_user_message('你好')
history.add_ai_message('python很有趣')
history.add_user_message('计算1+1=')

print(history.messages)

res=llm.invoke(history.messages)
print(res.content)


[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='python很有趣', additional_kwargs={}, response_metadata={}), HumanMessage(content='计算1+1=', additional_kwargs={}, response_metadata={})]
1 + 1 = 2。


二、ConversationBufferMemory

In [ ]:
#举例1 以字符串形式返回
from langchain.memory import ConversationBufferMemory

memroy=ConversationBufferMemory()

#inputs对应用户消息 outputs对应ai消息
memroy.save_context(inputs={'input':'你好 我叫小明'},outputs={'output':'很高兴认识你'})
memroy.save_context(inputs={'input':'帮我回答一下1+2等于几'},outputs={'output':'3'})

#返回字典的结构的key叫history
print(memroy.load_memory_variables({}))

In [ ]:
#举例2 以消息列表形式返回
from langchain.memory import ConversationBufferMemory

memroy=ConversationBufferMemory(return_messages=True)

memroy.save_context(inputs={'input':'你好 我叫小明'},outputs={'output':'很高兴认识你'})
memroy.save_context(inputs={'input':'帮我回答一下1+2等于几'},outputs={'output':'3'})

print(memroy.load_memory_variables({}))
print('\n')
print(memroy.chat_memory.messages)


In [ ]:
#举例3 结合llm,PromptTemplate

#提示词模板
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {history}
人类问题: {question}
回复:
"""
)
#memory
memory = ConversationBufferMemory()

#Chain 会使用memory中的history给提示词模板的history赋值
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

res = chain.invoke(input={'question': '你好 我叫小明'})

print(res)
print('\n')
res=chain.invoke(input={'question': '我的名字是什么'})
print(res)

In [ ]:
#举例4 基于举例3 修改memory_key

#提示词模板
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {chat}
人类问题: {question}
回复:
"""
)
#memory 修改memory_key为chat
memory = ConversationBufferMemory(memory_key='chat')


chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

res = chain.invoke(input={'question': '你好 我叫小明'})

print(res)
print('\n')
res=chain.invoke(input={'question': '我的名字是什么'})
print(res)

In [ ]:
#举例5 结合ChatPromptTemplate

from langchain_core.messages import SystemMessage
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate,HumanMessagePromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human","问题：{question}")
])


memory = ConversationBufferMemory(return_messages=True)

llm_chain = LLMChain(prompt=prompt,llm=llm, memory=memory)

#第一次调用就把问题和答复写入了history
res1 = llm_chain.invoke({"question": "中国首都在哪里？"})
print(res1,end="\n\n")


In [ ]:
res2 = llm_chain.invoke({"question": "我刚刚问了什么"})
print(res2)

三、ConversationChain的使用    将memory和Chain结合为一步 甚至可以结合提还差模板

In [ ]:
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {history}
人类问题: {input}
回复:
"""
)
# #memory
# memory = ConversationBufferMemory()
#
# #Chain 会使用memory中的history给提示词模板的history赋值
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

chain=ConversationChain(llm=llm, prompt=prompt_template)

res = chain.invoke(input={'input': '你好 我叫小明'})

print(res)

In [ ]:
res=chain.invoke(input={'input': '我的名字是什么'})
print(res)

In [ ]:
#内部也有默认的提示词模板 一个变量是input 另一个是history
chain=ConversationChain(llm=llm)
res = chain.invoke(input={'input': '今天星期三'})
print(res)
res=chain.invoke(input={'input':'明天星期几'})
print(res)

四、ConversationBufferWindowMemory

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# 1. 初始化消息存储（新版必须显式指定，用于存放历史消息）
chat_history = ChatMessageHistory()

# 2. 初始化窗口记忆（k=2 表示只保留最近2条消息）
memory = ConversationBufferWindowMemory(
    k=2,
    chat_memory=chat_history,  # 绑定消息存储
    return_messages=False  # 设为 False，返回文本格式的 history（而非消息对象列表）
)

chat_history.add_message(HumanMessage(content="你好 我叫小明"))
chat_history.add_message(AIMessage(content="很高兴认识你"))
chat_history.add_message(HumanMessage(content="帮我回答一下1+2等于几"))
chat_history.add_message(AIMessage(content="3"))
chat_history.add_message(HumanMessage(content="一周有几天"))
chat_history.add_message(AIMessage(content="7"))

print(memory.load_memory_variables({}))

In [ ]:
#结合llm
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话。
当前对话: {history}
人类问题: {input}
回复:
"""
)
#memory 只保留前一条消息
memory = ConversationBufferWindowMemory(k=1)

#Chain 会使用memory中的history给提示词模板的history赋值
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

res = chain.invoke(input={'input': '你好 我叫小明'})
print(res)
res=chain.invoke(input={'input':'今天星期三'})
print(res)
res=chain.invoke(input={'input':'明天星期几'})
print(res)
res=chain.invoke(input={'input':'我叫什么名字'})
print(res)